

# Hands-on Exercise 3 — Log a Model with a Signature
### AI Operations (AIOps) — MLflow Deep Dive | ~10–15 minutes

**Referenced in:** *MLflow Deep Dive Slide Deck*, Section 3 (MLflow Models)

**Objective:** attach a proper input/output **signature** and **input example** to a logged model,
then load it back with the framework-agnostic `pyfunc` interface and run inference on new data.

**Steps (from the slide deck):**
1. Using your Exercise 1 model, generate predictions on a small sample of training data.
2. Call `infer_signature()` to build a signature from that sample.
3. Re-log the model with `mlflow.sklearn.log_model(..., signature=..., input_example=...)`.
4. Open the run in the MLflow UI and confirm the signature appears under the Artifacts tab (`MLmodel` file).
5. Load the model back with `mlflow.pyfunc.load_model()` in a fresh Python session and call `.predict()` on new data.

**Deliverable:** a run containing a model artifact with a valid signature, plus a short script proving
`pyfunc.load_model()` + `.predict()` works end-to-end.

> **Prerequisite:** the MLflow Tracking Server must still be running at `http://localhost:5000`.

## Step 0 — Setup

In [29]:
# !pip install mlflow scikit-learn pandas --quiet
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("iris-classifier")

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 1 — Train a model and generate sample predictions

In [30]:
model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
model.fit(X_train, y_train)

sample_inputs = X_train.iloc[:5]
sample_predictions = model.predict(sample_inputs)
print(sample_inputs)
print("Sample predictions:", sample_predictions)

    sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)
22                4.6               3.6                1.0               0.2
15                5.7               4.4                1.5               0.4
65                6.7               3.1                4.4               1.4
11                4.8               3.4                1.6               0.2
42                4.4               3.2                1.3               0.2
Sample predictions: [0 0 1 0 0]


## Step 2 — Build a signature with `infer_signature()`
The signature records the expected input schema (column names + dtypes) and the output schema.

In [31]:
signature = infer_signature(sample_inputs, sample_predictions)
print(signature)

inputs: 
  ['sepal length (cm)': double (required), 'sepal width (cm)': double (required), 'petal length (cm)': double (required), 'petal width (cm)': double (required)]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None



## Step 3 — Log the model WITH the signature and an input example

In [32]:
with mlflow.start_run(run_name="mnist-mlp-with-signature") as run:
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)

    mlflow.log_param("n_estimators", 150)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("accuracy", acc)

    mlflow.sklearn.log_model(
    model,
    name="model",
    signature=signature,
    input_example=sample_inputs,
    serialization_format="pickle",  # <-- add this line
)

    signed_run_id = run.info.run_id

print(f"Logged run with signature: {signed_run_id}  (accuracy={acc:.4f})")

2026/08/30 16:57:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run mnist-mlp-with-signature at: http://localhost:5000/#/experiments/1/runs/49c6ae7adcec4274b0d1934b3dcdcce9
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run with signature: 49c6ae7adcec4274b0d1934b3dcdcce9  (accuracy=1.0000)


## Step 4 — Inspect the signature in the MLflow UI
1. Open **http://localhost:5000** → **iris-classifier** experiment → the `rf-with-signature` run.
2. Open the **Artifacts** tab → `model` → view the `MLmodel` file.
3. Confirm you can see a `signature:` block listing `inputs` and `outputs`.

You can also inspect it directly from Python without leaving the notebook:

In [33]:
from mlflow.models import get_model_info

model_uri = f"runs:/{signed_run_id}/model"
info = get_model_info(model_uri)
print("Signature recorded in MLmodel file:")
print(info.signature)

Signature recorded in MLmodel file:
inputs: 
  ['sepal length (cm)': double (required), 'sepal width (cm)': double (required), 'petal length (cm)': double (required), 'petal width (cm)': double (required)]
outputs: 
  [Tensor('int64', (-1,))]
params: 
  None



## Question 2 — MLP on MNIST, 6 runs varying learning rate + hidden layer size

In [34]:
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
import pandas as pd

mlflow.set_experiment("mnist-mlp-comparison")

mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
X_m, y_m = mnist.data, mnist.target.astype(int)

X_sub, _, y_sub, _ = train_test_split(X_m, y_m, train_size=8000, stratify=y_m, random_state=42)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_sub, y_sub, test_size=0.2, stratify=y_sub, random_state=42
)

scaler = StandardScaler()
X_train_m = scaler.fit_transform(X_train_m)
X_test_m = scaler.transform(X_test_m)

runs_config = [
    {"hidden_layer_sizes": (32,),     "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (32,),     "learning_rate_init": 0.01},
    {"hidden_layer_sizes": (64,),     "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (64,),     "learning_rate_init": 0.01},
    {"hidden_layer_sizes": (128, 64), "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (128, 64), "learning_rate_init": 0.01},
]

results = []
for i, cfg in enumerate(runs_config, start=1):
    with mlflow.start_run(run_name=f"mlp-mnist-run-{i}") as run:
        m = MLPClassifier(
            hidden_layer_sizes=cfg["hidden_layer_sizes"],
            learning_rate_init=cfg["learning_rate_init"],
            max_iter=100,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=42,
        )
        m.fit(X_train_m, y_train_m)

        train_acc = accuracy_score(y_train_m, m.predict(X_train_m))
        val_acc = accuracy_score(y_test_m, m.predict(X_test_m))
        final_loss = m.loss_curve_[-1]

        mlflow.log_param("hidden_layer_sizes", cfg["hidden_layer_sizes"])
        mlflow.log_param("learning_rate_init", cfg["learning_rate_init"])
        mlflow.log_param("dataset", "MNIST")
        mlflow.log_param("model_type", "MLPClassifier")

        mlflow.log_metric("train_accuracy", train_acc)
        mlflow.log_metric("val_accuracy", val_acc)
        mlflow.log_metric("train_loss", final_loss)
        for epoch, loss in enumerate(m.loss_curve_):
            mlflow.log_metric("train_loss_curve", loss, step=epoch)

        sig = infer_signature(X_train_m[:5], m.predict(X_train_m[:5]))
        mlflow.sklearn.log_model(m, name="model", signature=sig,serialization_format="pickle")

        results.append({
            "run_id": run.info.run_id,
            "hidden_layer_sizes": cfg["hidden_layer_sizes"],
            "learning_rate_init": cfg["learning_rate_init"],
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "train_loss": final_loss,
        })
        print(f"Run {i}: hidden={cfg['hidden_layer_sizes']}, lr={cfg['learning_rate_init']} "
              f"-> val_acc={val_acc:.4f}, train_loss={final_loss:.4f}")

pd.DataFrame(results).sort_values("val_accuracy", ascending=False)

2026/08/30 16:57:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 1: hidden=(32,), lr=0.001 -> val_acc=0.9269, train_loss=0.0165
🏃 View run mlp-mnist-run-1 at: http://localhost:5000/#/experiments/2/runs/2fd7ea9131be476da9cc43790ebea7f2
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/08/30 16:58:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 2: hidden=(32,), lr=0.01 -> val_acc=0.9287, train_loss=0.0007
🏃 View run mlp-mnist-run-2 at: http://localhost:5000/#/experiments/2/runs/a90b89da8c7c453b8085702c98ba4a2f
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/08/30 16:58:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

2026/08/30 16:58:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 5: hidden=(128, 64), lr=0.001 -> val_acc=0.9344, train_loss=0.0041
🏃 View run mlp-mnist-run-5 at: http://localhost:5000/#/experiments/2/runs/d350b911167c461297c616a71c718c96
🧪 View experiment at: http://localhost:5000/#/experiments/2


2026/08/30 16:58:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 6: hidden=(128, 64), lr=0.01 -> val_acc=0.9394, train_loss=0.2544
🏃 View run mlp-mnist-run-6 at: http://localhost:5000/#/experiments/2/runs/2d6788dee48b4830ada5ad4756c87c76
🧪 View experiment at: http://localhost:5000/#/experiments/2


,run_id,hidden_layer_sizes,learning_rate_init,train_accuracy,val_accuracy,train_loss
3,d7180f658611472c80b623b2508101c1,"(64,)",0.010,0.989531,0.939375,0.000665
5,2d6788dee48b4830ada5ad4756c87c76,"(128, 64)",0.010,0.989062,0.939375,0.254446
4,d350b911167c461297c616a71c718c96,"(128, 64)",0.001,0.989062,0.934375,0.004107
2,971158ef51954527b82fb79ed87a5cd0,"(64,)",0.001,0.987344,0.931875,0.013424
1,a90b89da8c7c453b8085702c98ba4a2f,"(32,)",0.010,0.987969,0.928750,0.000652
0,2fd7ea9131be476da9cc43790ebea7f2,"(32,)",0.001,0.985000,0.926875,0.016505


## Step 5 — Load the model with `pyfunc` and predict on new data
This simulates a *fresh Python session* consuming the model purely through its MLflow URI — no need to import scikit-learn model internals.

In [35]:
import mlflow.pyfunc

loaded_model = mlflow.pyfunc.load_model(model_uri)

new_data = X_test.iloc[:8]
predictions = loaded_model.predict(new_data)
print("Predictions from the reloaded pyfunc model:")
print(predictions)

Predictions from the reloaded pyfunc model:
[1 0 2 1 1 0 1 2]


### Try it: what happens if the input doesn't match the signature?
Uncomment and run the cell below to see MLflow's schema validation in action (this previews Exercise 5's serving validation behaviour).

In [36]:
# bad_input = new_data.rename(columns={"sepal length (cm)": "sepal_length"})  # wrong column name
# loaded_model.predict(bad_input)  # -> raises a schema validation error

---
### ✅ Deliverable checklist
- [ ] A run (`rf-with-signature`) whose logged model has a non-empty `signature` in its `MLmodel` file
- [ ] An `input_example` visible alongside the model artifact
- [ ] A working call to `mlflow.pyfunc.load_model(...)` followed by `.predict()` on new data, executed in this notebook
- [ ] (Optional) A screenshot of the signature block from the MLflow UI